In [1]:
import numpy as np
from faker import Faker
import random
import random
import numpy as np
from faker import Faker
import psycopg
from pgvector.psycopg import register_vector

In [2]:
fake = Faker()

In [3]:
documents = []

for i in range(100):
    doc = {
        "id": i + 1,
        "text": fake.sentence(nb_words=random.randint(5, 15)),
        "vector": np.random.rand(768).tolist()  # vector random float [0,1)
    }
    documents.append(doc)

In [4]:
len(documents[0]['vector'])

768

In [5]:
print(len(documents))

100


In [6]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="my_db",
    user="postgres",
    password="secret", 
)

In [7]:
register_vector(conn)

In [8]:
with conn.cursor() as cur:
    documents = []
    for _ in range(100):
        content = fake.sentence(nb_words=random.randint(5, 15))
        embedding = np.random.rand(768).tolist()
        documents.append((content, embedding))

    cur.executemany(
        """
        INSERT INTO documents (content, embedding)
        VALUES (%s, %s)
        """,
        documents,
    )

    conn.commit()

In [9]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM documents;")
    total_docs = cur.fetchone()[0]
    print(f"Total documents: {total_docs}")

Total documents: 100


In [10]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT id, content, embedding
        FROM documents
        LIMIT 1;
    """)

    row = cur.fetchone()

    if row:
        doc_id, content, embedding = row

        print("ID:", doc_id)
        print("Content:", content)
        print("Embedding length:", len(embedding))  # harusnya 768
        print("Embedding sample:", embedding[:5])   # lihat 5 angka pertama
    else:
        print("No documents found")    

ID: 1
Content: Current force represent seek while alone.
Embedding length: 768
Embedding sample: [0.40890947 0.6533762  0.12992652 0.3721229  0.4215365 ]


In [11]:
len(embedding)

768

In [12]:
query_embedding = np.random.rand(768).tolist()

In [13]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            id,
            content,
            embedding <=> %s::vector AS distance
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT 5;
        """,
        (query_embedding, query_embedding)
    )

    results = cur.fetchall()

    print("Top 5 similar documents:")
    for row in results:
        doc_id, content, distance = row

        print(f"ID: {doc_id}")
        print(f"Content: {content}")
        print(f"Distance: {distance:.4f}")
        print("-" * 50)

Top 5 similar documents:
ID: 95
Content: Back check arm year responsibility half performance bill physical issue thought federal yard left party.
Distance: 0.2260
--------------------------------------------------
ID: 64
Content: Way necessary business various sort possible charge produce.
Distance: 0.2342
--------------------------------------------------
ID: 31
Content: Develop letter lay property check trouble operation car occur increase.
Distance: 0.2375
--------------------------------------------------
ID: 58
Content: Ago industry since forward identify medical bed morning.
Distance: 0.2382
--------------------------------------------------
ID: 3
Content: Watch society well head worry into institution.
Distance: 0.2393
--------------------------------------------------


In [14]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            id,
            content,
            embedding <-> %s::vector AS distance
        FROM documents
        ORDER BY embedding <-> %s::vector
        LIMIT 5;
        """,
        (query_embedding, query_embedding)
    )

    results = cur.fetchall()

    print("Top 5 similar documents:")
    for row in results:
        doc_id, content, distance = row

        print(f"ID: {doc_id}")
        print(f"Content: {content}")
        print(f"Distance: {distance:.4f}")
        print("-" * 50)

Top 5 similar documents:
ID: 95
Content: Back check arm year responsibility half performance bill physical issue thought federal yard left party.
Distance: 10.8226
--------------------------------------------------
ID: 58
Content: Ago industry since forward identify medical bed morning.
Distance: 10.9495
--------------------------------------------------
ID: 64
Content: Way necessary business various sort possible charge produce.
Distance: 10.9977
--------------------------------------------------
ID: 26
Content: Reveal action stuff stand stay soon animal night per.
Distance: 11.0254
--------------------------------------------------
ID: 69
Content: Quite security traditional after president feel bar top.
Distance: 11.0438
--------------------------------------------------
